# Analysis

In [3]:
import pandas as pd
import numpy as np

In [4]:
ekf = pd.read_csv("./hasil_sementara_21_sep/ekf_flight_data_figure8.csv")
ekf.head()

,time_sec,true_mass_kg,true_offset_x_m,true_offset_y_m,actual_x_m,actual_y_m,actual_z_m
0,0.000000,7.8,0.15,0.15,-0.000005,-0.000005,0.198577
1,0.041667,7.8,0.15,0.15,-0.000010,-0.000010,0.188605
2,0.083333,7.8,0.15,0.15,0.000059,0.000058,0.184471
3,0.125000,7.8,0.15,0.15,0.000227,0.000225,0.182779
4,0.166667,7.8,0.15,0.15,0.000543,0.000539,0.182933


In [5]:
ekf.tail()

,time_sec,true_mass_kg,true_offset_x_m,true_offset_y_m,actual_x_m,actual_y_m,actual_z_m
2876,119.833333,7.8,0.15,0.15,2.255378,-3.275836,6.192379
2877,119.875000,7.8,0.15,0.15,2.161400,-3.206948,6.166030
2878,119.916667,7.8,0.15,0.15,2.067448,-3.137142,6.139679
2879,119.958333,7.8,0.15,0.15,1.973543,-3.066434,6.113327
2880,120.000000,7.8,0.15,0.15,1.879701,-2.994837,6.086975


In [6]:
def show_evaluation(
    time_sec, 
    true_mass_kg, est_mass_kg, 
    true_offset_x_m, est_offset_x_m, 
    true_offset_y_m, est_offset_y_m, 
    actual_x_m, target_x_m, 
    actual_y_m, target_y_m, 
    actual_z_m, target_z_m
):
    # Convert lists/pandas columns to numpy arrays for fast matrix math
    time_sec = np.array(time_sec)
    
    # 1. Trajectory Tracking Metrics
    errors_x = np.array(actual_x_m) - np.array(target_x_m)
    errors_y = np.array(actual_y_m) - np.array(target_y_m)
    errors_z = np.array(actual_z_m) - np.array(target_z_m)
    errors_3d = np.column_stack((errors_x, errors_y, errors_z))

    mae_xyz = np.mean(np.abs(errors_3d), axis=0)
    rmse_xyz = np.sqrt(np.mean(errors_3d**2, axis=0))

    distances = np.linalg.norm(errors_3d, axis=1)
    mae_3d = np.mean(distances)
    rmse_3d = np.sqrt(np.mean(distances**2))

    # 2. PINN Parameter Estimation Metrics
    error_mass = np.array(est_mass_kg) - np.array(true_mass_kg)
    error_cx = np.array(est_offset_x_m) - np.array(true_offset_x_m)
    error_cy = np.array(est_offset_y_m) - np.array(true_offset_y_m)

    # RMSE for the entire flight
    rmse_mass = np.sqrt(np.mean(error_mass**2))
    rmse_cx = np.sqrt(np.mean(error_cx**2))
    rmse_cy = np.sqrt(np.mean(error_cy**2))
    
    # Final absolute error (to check if the estimator successfully converged by the end)
    final_mass_err = np.abs(error_mass[-1])
    final_cx_err = np.abs(error_cx[-1])
    final_cy_err = np.abs(error_cy[-1])

    # Calculate duration safely
    simulation_duration = time_sec[-1] - time_sec[0] if len(time_sec) > 0 else 0.0

    # 3. Print the Final Report
    print("\n" + "="*45)
    print("      CONTROLLER & PINN METRICS      ")
    print("="*45)
    print(f"DURATION: {simulation_duration:.2f} seconds")
    print("="*45)
    print("[TRAJECTORY TRACKING]")
    print(f" 3D Euclidean MAE:    {mae_3d:.4f} m")
    print(f" 3D Euclidean RMSE:   {rmse_3d:.4f} m")
    print(f" -> X-Axis RMSE:      {rmse_xyz[0]:.4f} m")
    print(f" -> Y-Axis RMSE:      {rmse_xyz[1]:.4f} m")
    print(f" -> Z-Axis RMSE:      {rmse_xyz[2]:.4f} m")
    print("-" * 45)
    print("[PINN PARAMETER ESTIMATION (OVERALL RMSE)]")
    print(f" Mass RMSE:           {rmse_mass:.4f} kg")
    print(f" CoG X-Offset RMSE:   {rmse_cx:.4f} m")
    print(f" CoG Y-Offset RMSE:   {rmse_cy:.4f} m")
    print("-" * 45)
    print("[PINN FINAL CONVERGENCE ERROR (LAST STEP)]")
    print(f" Final Mass Error:    {final_mass_err:.4f} kg")
    print(f" Final CoG X Error:   {final_cx_err:.4f} m")
    print(f" Final CoG Y Error:   {final_cy_err:.4f} m")
    print("="*45 + "\n")

In [7]:
show_evaluation(
    time_sec=ekf['time_sec'],
    
    true_mass_kg=ekf['true_mass_kg'], 
    est_mass_kg=ekf['est_mass_kg'], 
    
    true_offset_x_m=ekf['true_offset_x_m'], 
    est_offset_x_m=ekf['est_offset_x_m'], 
    
    true_offset_y_m=ekf['true_offset_y_m'], 
    est_offset_y_m=ekf['est_offset_y_m'], 
    
    actual_x_m=ekf['actual_x_m'], 
    target_x_m=ekf['target_x_m'], 
    
    actual_y_m=ekf['actual_y_m'], 
    target_y_m=ekf['target_y_m'], 
    
    actual_z_m=ekf['actual_z_m'], 
    target_z_m=ekf['target_z_m']
)

KeyError: 'est_mass_kg'